# Question 1


In [19]:
import pandas as pd
import numpy as np

"""
Question 1

Join the three datasets: Energy, GDP, and ScimEn into a new dataset (using the intersection of country names). Use only the 10 years (2006-2015) of GDP data and only the top 15 countries by Scimagojr 'Rank' (Rank 1 through 15).
The index of this DataFrame should be the name of the country, and the columns should be
['Rank', 'Documents', 'Citable documents', 'Citations', 'Self-citations', 'Citations per document', 'H index', 'Energy Supply', 'Energy Supply per Capita', '% Renewable', '2006', '2007', '2008', '2009', '2010', '2011', 2012', '2013', '2014', '2015']
Function "answer_one" should return the resulted DataFrame (20 columns and 15 entries)
"""


class Question1:

    def _query_energy(self):
        energy = pd.read_excel('Energy.xlsx', header=None,
                               names=['Country', 'Energy Supply', 'Energy Supply per Capita', '% Renewable'],
                               usecols='C:F', skiprows=18, nrows=228, engine='openpyxl')

        energy = energy.replace(['...'], np.nan)
        energy['Energy Supply'] *= 1000000

        source_countries_names = ['Republic of Korea', 'United States of America20',
                                  'United Kingdom of Great Britain and Northern Ireland',
                                  'China, Hong Kong Special Administrative Region',
                                  'Bolivia (Plurinational State of)', 'Switzerland17']
        final_countries_names = ['South Korea', 'United States', 'United Kingdom',
                                 'Hong Kong', 'Bolivia', 'Switzerland']
        energy['Country'] = energy['Country'].replace(source_countries_names, final_countries_names)

        return energy

    def _query_gdp(self):
        gdp = pd.read_csv('world_bank.csv', sep=',', header=0,
                          dtype=None, skiprows=4, encoding='utf-8')

        scn_gdp = ['Korea, Rep.', 'Iran, Islamic Rep.', 'Hong Kong SAR, China']  # source country names
        fcn_gdp = ['South Korea', 'Iran', 'Hong Kong']  # final country names
        gdp['Country'] = gdp['Country'].replace(scn_gdp, fcn_gdp)

        return gdp

    def _query_scimen(self):
        scim_en = pd.read_excel('scimEn.xlsx', header=0,
                                engine='openpyxl')
        return scim_en


    def answer_one(self):
        energy = self._query_energy()
        gdp = self._query_gdp()
        scim_en = self._query_scimen()

        energy.set_index('Country', inplace=True)
        gdp.set_index('Country', inplace=True)
        scim_en.set_index('Country', inplace=True)

        temp_df = pd.merge(scim_en, energy, on='Country', how='left')
        final_df = pd.merge(temp_df, gdp, on='Country', how='left')

        final_df.drop(final_df.tail(196).index, inplace=True)
        final_df.drop(final_df.iloc[:, 1:2], axis=1, inplace=True)
        final_df.drop(final_df.iloc[:, 10:58], axis=1, inplace=True)
        final_df.drop(final_df.iloc[:, 21:28], axis=1, inplace=True)

        # print(final_df.to_string())
        return final_df


if __name__ == "__main__":
    question1 = Question1()
    qa1 = question1.answer_one()
    print(qa1)
    # qa1.to_excel('answer1.xlsx')


                    Rank  Documents  Citable documents  Citations  \
Country                                                             
China                  1     303064             301778    3036531   
United States          2     184851             181106    2623922   
India                  3      60257              58589     590570   
Japan                  4      52780              52281     557023   
United Kingdom         5      47141              45928     748994   
Germany                6      42343              41464     528645   
Russian Federation     7      39424              39189     142937   
Canada                 8      35588              34940     665415   
Italy                  9      31260              29959     433388   
South Korea           10      31200              30949     405923   
France                11      26848              26320     403834   
Iran                  12      25481              25204     398571   
Spain                 13      2496

# Question 2



In [20]:
"""
Question 2
What is the average GDP over the last 10 years for each country? (exclude missing values from this calculation.)

This function should return a Series named avgGDP with 15 countries and their average GDP sorted in descending order.
"""


def answer_two():
    question1 = Question1()
    top15 = question1.answer_one()
    # print(top15.to_string(), '\n')

    avgGDP = pd.Series(dtype='float64')
    n = len(top15.iloc[:, 1])
    for i in range(n):
        # TODO: what's better: make more understandable variables or less lines of code?
        # TODO: why this code doesn't work in jupyter
        country = tuple((top15.index[i],))
        avg = top15.iloc[i, 10:].mean()
        new_element = pd.Series([avg], index=country)
        avgGDP = pd.concat([avgGDP, new_element])

    # print(avgGDP.to_string(), '\n')
    return avgGDP


if __name__ == "__main__":
    a2 = answer_two()
    print(a2)
    a2.to_excel("answer2.xlsx")


China                 6.505726e+12
United States         1.546177e+13
India                 1.531264e+12
Japan                 5.202535e+12
United Kingdom        2.769540e+12
Germany               3.461844e+12
Russian Federation    1.584680e+12
Canada                1.576064e+12
Italy                 2.117098e+12
South Korea           1.195329e+12
France                2.646392e+12
Iran                  4.354285e+11
Spain                 1.378271e+12
Brazil                1.889139e+12
Australia             1.160558e+12
dtype: float64


# Question 3



In [21]:
"""
Question 3
By how much had the GDP changed over the 10-year span for the country with the 6th largest average GDP?

This function should return a single number.
"""


def answer_three():
    question1 = Question1()

    avg_gdp = answer_two()  # Series of avg_gdp
    avg_gdp = avg_gdp.sort_values(ascending=True)
    sixth_gdp_country = avg_gdp.index[5]  # name of sixth country
    gdps_data = question1.answer_one().loc[sixth_gdp_country]
    # print(gdps_data.to_string(), '\n')
    # TODO: Do I have to return absolute value or change in percent?
    balance = gdps_data.loc['2015'] - gdps_data.loc['2005']

    return balance


if __name__ == "__main__":
    print(answer_three())


383400217438.45996


# Question 4


In [22]:
"""
Create a new column that is the ratio of Self-Citations to Total Citations.
What is the maximum value for this new column, and what country has the highest ratio?

This function should return a tuple with the name of the country and the ratio.
"""


def answer_four():
    question1 = Question1()
    top15 = question1.answer_one()

    country = ''
    max_ratio = 0
    n = len(top15.iloc[:, 1])
    for i in range(n):
        ratio = top15.iloc[i, 4] / top15.iloc[i, 3]
        if ratio > max_ratio:
            country = top15.index[i]
            max_ratio = ratio

    return tuple((country, max_ratio),)


if __name__ == "__main__":
    print(answer_four())


('China', 0.689186772669207)


# Question 5


In [23]:
"""
Create a column that estimates the population using Energy Supply and Energy Supply per capita. 
What is the third most populous country according to this estimate?

This function should return a single string value.
"""


def answer_five():
    question1 = Question1()
    top15 = question1.answer_one()

    population_list = []
    n = len(top15.iloc[:, 1])
    for i in range(n):
        population = top15.iloc[i, 7] / top15.iloc[i, 8]
        if population > 0:
            population_list.append([top15.index[i], int(population)])

    # print(population_list, '\n')
    population_list.sort()
    temp = sorted(population_list, key=lambda population_list: population_list[1])
    # print(temp, '\n')

    return temp[-3][0]


if __name__ == "__main__":
    print(answer_five())


Brazil


# Question 6


In [24]:
"""
Create a column that estimates the number of citable documents per person.
What is the correlation between the number of citable documents per capita
and the energy supply per capita?
Use the .corr() method, (Pearson's correlation).

This function should return a single number.
"""


def answer_six():
    question1 = Question1()
    top15 = question1.answer_one()

    df = pd.DataFrame(columns=['docs_per_capita', 'energy_supply_per_capita'])
    n = len(top15.iloc[:, 1])
    for i in range(n):
        population = top15.iloc[i, 7] / top15.iloc[i, 8]
        if population > 0:
            docs_per_capita = top15.iloc[i, 2] / population
            energy_supply_per_capita = top15.iloc[i, 8]
            temp_df = pd.DataFrame({'docs_per_capita': [docs_per_capita],
                                    'energy_supply_per_capita': [energy_supply_per_capita]})
            # print(temp_df.to_string(), '\n')
            df = pd.concat([df, temp_df])

    # print(df.to_string(), '\n')

    return df.corr().iloc[0, 1]


if __name__ == "__main__":
    print(answer_six())


0.8623303885077127


# Question 7


In [28]:
"""
Use the following dictionary to group the Countries by Continent, 
then create a dateframe that displays the sample size 
(the number of countries in each continent bin), 
and the sum, mean, and std deviation for the estimated population of each country.
"""


def answer_seven(ContinentDict):
    question1 = Question1()
    top15 = question1.answer_one()

    zeros = [0, 0, 0, 0, 0]
    countries_population = pd.DataFrame(index=['Asia', 'Australia', 'Europe',
                                               'North America', 'South America'],
                                        columns=ContinentDict.keys())
    final_df = pd.DataFrame({'Continent': ['Asia', 'Australia', 'Europe',
                                           'North America', 'South America'],
                             'size': zeros, 'sum': zeros,
                             'mean': zeros, 'std': zeros}).set_index(['Continent'])
    # print(final_df.to_string(), '\n')
    for i in ContinentDict:
        row_index = list(ContinentDict).index(i)
        population = top15.iloc[row_index, 7] / top15.iloc[row_index, 8]

        if population > 0:
            row_index = ContinentDict[i]
            countries_population.loc[row_index, i] = population
            final_df.loc[row_index, 'size'] += 1
            final_df.loc[row_index, 'sum'] += population
            final_df.loc[row_index, 'mean'] += final_df.loc[row_index, 'sum'] / final_df.loc[row_index, 'size']

    # print(final_df.to_string(), '\n')
    # print(countries_population.to_string(), '\n')
    n = len(countries_population.iloc[:, 1])
    for j in range(n):
        final_df.iloc[j, 3] = countries_population[list(ContinentDict.keys())].std(axis=1, skipna=True)[j]

    return final_df


if __name__ == "__main__":
    ContinentDict = {'China': 'Asia',
                     'United States': 'North America',
                     'Japan': 'Asia',
                     'United Kingdom': 'Europe',
                     'Russian Federation': 'Europe',
                     'Canada': 'North America',
                     'Germany': 'Europe',
                     'India': 'Asia',
                     'France': 'Europe',
                     'South Korea': 'Asia',
                     'Italy': 'Europe',
                     'Spain': 'Europe',
                     'Iran': 'Asia',
                     'Australia': 'Australia',
                     'Brazil': 'South America'}
    
    answer = answer_seven(ContinentDict)
    answer.to_excel('answer7.xlsx')
    print(answer)

               size           sum          mean           std
Continent                                                    
Asia              3  1.361776e+09  2.386641e+09  7.126076e+08
Australia         1  2.059153e+08  2.059153e+08           NaN
Europe            1  1.435000e+08  1.435000e+08           NaN
North America     2  3.979851e+08  5.166079e+08  1.677580e+08
South America     0  0.000000e+00  0.000000e+00           NaN
